# TD3-02 — Confirmación independiente de TD3 frente a DDPG-H02-R2

**Objetivo científico:** comprobar con cinco **semillas de entrenamiento nuevas** (1301–1305) si la señal del piloto TD3-01 se reproduce. **DDPG continúa siendo el núcleo de la tesis**; TD3 es una comparación algorítmica adicional.

**Comparación congelada:** modelo Cart-Pole **idéntico** al usado en TD3-01/N5.1, recompensa R2 sin cambios, 4 observaciones `[x,v,θ,ω]`, acción en ±10 N, redes `[128,128]`, `learning_rate=3e-4`, `gamma=.995`, `tau=.010`, `batch_size=128`, ruido exploratorio `sigma=.10`, **160 000 pasos por política**; TD3 incorpora sus dos críticos, `policy_delay=2` y suavizado del objetivo (`target_policy_noise=.20`, `target_noise_clip=.50`). No ajustar parámetros después de ver resultados.

**Nuevos conjuntos excluyentes:** entrenamiento 1301–1305, validación 8400–8404, prueba final 10010–10019. Checkpoints 40K/80K/120K/160K; la selección se realiza **únicamente con validación**. La prueba final se ejecuta después y no debe usarse para cambiar la selección.

**Recuperación ante fallas:** cada *rama completa* se preserva en Drive. Las ramas completas se omiten al ejecutar de nuevo; si Colab se desconecta a mitad de una rama, ésta se entrena nuevamente desde cero (la carga exclusiva de un archivo de pesos **no** reproduce el estado del replay buffer/optimizadores/RNG).

**Significancia:** cinco semillas nuevas son una réplica útil, pero una prueba exacta bilateral de signos/inversión con cinco pares no alcanza `p<.05` aunque todas las diferencias tengan el mismo signo. El informe debe distinguir magnitud, consistencia e incertidumbre. No equiparar los 50 episodios de evaluación por algoritmo con 50 entrenamientos independientes.


## 1. Instalación (ejecutar antes de importar bibliotecas)

In [ ]:
%pip -q install 'stable-baselines3>=2.3,<3' 'gymnasium>=0.29,<2' 'pandas>=2' 'matplotlib>=3'

## 2. Configuración y ruta persistente en Google Drive

In [ ]:
from pathlib import Path
import os, math, time, random, json, hashlib, platform, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import TD3, DDPG
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed
from google.colab import drive, files

drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/TD3_02_VALIDACION_INDEPENDIENTE_DDPG_R2')
OUT.mkdir(parents=True,exist_ok=True)
assert OUT.is_dir() and str(OUT.resolve()).startswith('/content/drive/MyDrive/')
probe=OUT/'_PRUEBA_ESCRITURA_DRIVE.tmp'
probe.write_text('TD3-02 OK',encoding='utf8')
assert probe.read_text(encoding='utf8')=='TD3-02 OK'
probe.unlink()
print('PRUEBAS OK: escritura y lectura reales en Google Drive.')
CFG=dict(train_seeds=[1301,1302,1303,1304,1305],total_steps=160000,
    checkpoints=[40000,80000,120000,160000],learning_starts=1000,
    val_seeds=list(range(8400,8405)),test_seeds=list(range(10010,10020)),
    episode_steps=6000,dt=.02,
    theta_bono_rad=.15,omega_bono_r1_rad_s=.50,
    x_bono_r2_m=.35,v_bono_r2_m_s=.25,bono_r2=3.,
    x_estricto=.20,v_estricto=.10,theta_estricto=.05,omega_estricto=.10)
H02=dict(learning_rate=3e-4,gamma=.995,tau=.010,batch_size=128,sigma=.10,net_arch=[128,128])
TD3_SPEC=dict(policy_delay=2,target_policy_noise=.20,target_noise_clip=.50)
ALGOS=('DDPG_H02_R2','TD3_H02_R2')
assert len(set(CFG['train_seeds']+CFG['val_seeds']+CFG['test_seeds']))==len(CFG['train_seeds']+CFG['val_seeds']+CFG['test_seeds'])
assert CFG['checkpoints'][-1]==CFG['total_steps']
print('Salida persistente:',OUT)
print('SB3:',sb3.__version__,'Gymnasium:',gym.__version__)
print('10 modelos desde cero; pasos totales:',len(ALGOS)*len(CFG['train_seeds'])*CFG['total_steps'])

## 3. Dinámica Cart-Pole original, copiada **sin modificar** del experimento N5.1

In [ ]:
from pathlib import Path
Path('envs').mkdir(exist_ok=True)
Path('envs/__init__.py').write_text('',encoding='utf-8')
ORIGINAL_ENV = '"""\nEntorno Gymnasium: péndulo invertido tipo carro–poste con acción continua.\n\nEstado:\n    s_t = [x_t, x_dot_t, theta_t, theta_dot_t]\n\nAcción:\n    a_t = u(t), fuerza continua aplicada al carro.\n\nConvención angular:\n    theta = 0  -> péndulo vertical hacia arriba.\n    theta = pi -> péndulo hacia abajo.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport numpy as np\n\n\nclass CartPoleContinuousEnv(gym.Env):\n    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}\n\n    def __init__(\n        self,\n        render_mode: Optional[str] = None,\n        max_episode_steps: int = 1000,\n        force_max: float = 10.0,\n        dt: float = 0.02,\n        x_threshold: float = 2.4,\n        noise_std: float = 0.0,\n        viscous_friction_cart: float = 0.0,\n        viscous_friction_pole: float = 0.0,\n        external_impulse: float = 0.0,\n        impulse_step: Optional[int] = None,\n        random_initial_state: bool = True,\n        start_near_downward: bool = True,\n    ):\n        super().__init__()\n\n        self.gravity = 9.81\n        self.masscart = 1.0\n        self.masspole = 0.1\n        self.total_mass = self.masscart + self.masspole\n        self.length = 0.5\n        self.polemass_length = self.masspole * self.length\n\n        self.force_max = float(force_max)\n        self.dt = float(dt)\n        self.x_threshold = float(x_threshold)\n        self.max_episode_steps = int(max_episode_steps)\n\n        self.noise_std = float(noise_std)\n        self.viscous_friction_cart = float(viscous_friction_cart)\n        self.viscous_friction_pole = float(viscous_friction_pole)\n        self.external_impulse = float(external_impulse)\n        self.impulse_step = impulse_step\n\n        self.random_initial_state = random_initial_state\n        self.start_near_downward = start_near_downward\n\n        self.render_mode = render_mode\n        self.screen = None\n        self.clock = None\n        self.screen_width = 800\n        self.screen_height = 500\n\n        self.action_space = spaces.Box(\n            low=np.array([-self.force_max], dtype=np.float32),\n            high=np.array([self.force_max], dtype=np.float32),\n            dtype=np.float32,\n        )\n\n        high = np.array(\n            [\n                self.x_threshold * 2.0,\n                np.finfo(np.float32).max,\n                math.pi,\n                np.finfo(np.float32).max,\n            ],\n            dtype=np.float32,\n        )\n        self.observation_space = spaces.Box(-high, high, dtype=np.float32)\n\n        self.state: Optional[np.ndarray] = None\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n    @staticmethod\n    def angle_normalize(theta: float) -> float:\n        return ((theta + math.pi) % (2.0 * math.pi)) - math.pi\n\n    def _get_obs(self) -> np.ndarray:\n        assert self.state is not None\n        obs = self.state.copy()\n\n        if self.noise_std > 0:\n            noise = self.np_random.normal(0.0, self.noise_std, size=obs.shape)\n            obs = obs + noise.astype(np.float32)\n\n        obs[2] = self.angle_normalize(float(obs[2]))\n        return obs.astype(np.float32)\n\n    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):\n        super().reset(seed=seed)\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n        if self.random_initial_state:\n            if self.start_near_downward:\n                x = self.np_random.uniform(low=-0.05, high=0.05)\n                x_dot = self.np_random.uniform(low=-0.05, high=0.05)\n                theta = math.pi + self.np_random.uniform(low=-0.10, high=0.10)\n                theta_dot = self.np_random.uniform(low=-0.05, high=0.05)\n            else:\n                x = self.np_random.uniform(low=-0.5, high=0.5)\n                x_dot = self.np_random.uniform(low=-0.5, high=0.5)\n                theta = self.np_random.uniform(low=-math.pi, high=math.pi)\n                theta_dot = self.np_random.uniform(low=-1.0, high=1.0)\n        else:\n            x, x_dot, theta, theta_dot = 0.0, 0.0, math.pi, 0.0\n\n        theta = self.angle_normalize(theta)\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n\n        info = {\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n        }\n        return self._get_obs(), info\n\n    def step(self, action):\n        assert self.state is not None\n\n        x, x_dot, theta, theta_dot = map(float, self.state)\n        force = float(np.clip(action[0], -self.force_max, self.force_max))\n\n        if self.impulse_step is not None and self.steps == self.impulse_step:\n            force += self.external_impulse\n\n        force -= self.viscous_friction_cart * x_dot\n\n        costheta = math.cos(theta)\n        sintheta = math.sin(theta)\n\n        temp = (\n            force + self.polemass_length * theta_dot**2 * sintheta\n        ) / self.total_mass\n\n        theta_acc = (\n            self.gravity * sintheta\n            - costheta * temp\n            - self.viscous_friction_pole * theta_dot / max(self.polemass_length, 1e-8)\n        ) / (\n            self.length * (4.0 / 3.0 - self.masspole * costheta**2 / self.total_mass)\n        )\n\n        x_acc = temp - self.polemass_length * theta_acc * costheta / self.total_mass\n\n        x_dot = x_dot + self.dt * x_acc\n        x = x + self.dt * x_dot\n        theta_dot = theta_dot + self.dt * theta_acc\n        theta = theta + self.dt * theta_dot\n        theta = self.angle_normalize(theta)\n\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n        self.steps += 1\n\n        reward = self._reward(x, x_dot, theta, theta_dot, force)\n\n        terminated = bool(abs(x) > self.x_threshold)\n        truncated = bool(self.steps >= self.max_episode_steps)\n\n        self.last_u = force\n        self.cumulative_control_effort += force**2 * self.dt\n        self.cumulative_reward += reward\n\n        info = {\n            "x": x,\n            "x_dot": x_dot,\n            "theta": theta,\n            "theta_dot": theta_dot,\n            "u": force,\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n            "is_upright": abs(theta) < 0.15,\n        }\n\n        if self.render_mode == "human":\n            self.render()\n\n        return self._get_obs(), float(reward), terminated, truncated, info\n\n    def _reward(self, x: float, x_dot: float, theta: float, theta_dot: float, u: float) -> float:\n        """\n        Recompensa mejorada para swing-up y estabilización.\n        """\n\n        reward = 1.0 + math.cos(theta)\n\n        reward += 0.05\n\n        reward -= 0.20 * x**2\n        reward -= 0.01 * x_dot**2\n        reward -= 0.02 * theta_dot**2\n        reward -= 0.001 * u**2\n\n        if abs(theta) < 0.15:\n            reward += 3.0\n\n        if abs(x) > 0.80 * self.x_threshold:\n            reward -= 5.0 * (abs(x) - 0.80 * self.x_threshold)\n\n        if abs(x) > self.x_threshold:\n            reward -= 100.0\n\n        return reward\n\n    def render(self):\n        try:\n            import pygame\n            from pygame import gfxdraw\n        except ImportError as exc:\n            raise ImportError("Para renderizar instala pygame: pip install pygame") from exc\n\n        if self.screen is None:\n            pygame.init()\n            if self.render_mode == "human":\n                self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))\n            else:\n                self.screen = pygame.Surface((self.screen_width, self.screen_height))\n            self.clock = pygame.time.Clock()\n\n        assert self.state is not None\n        x, _, theta, _ = self.state\n\n        world_width = self.x_threshold * 2\n        scale = self.screen_width / world_width\n        cart_y = int(self.screen_height * 0.65)\n        cart_w = 60\n        cart_h = 30\n        pole_len = scale * (2 * self.length)\n\n        cart_x = int(self.screen_width / 2 + x * scale)\n\n        self.screen.fill((255, 255, 255))\n        import pygame\n\n        pygame.draw.line(\n            self.screen, (0, 0, 0),\n            (0, cart_y + cart_h // 2),\n            (self.screen_width, cart_y + cart_h // 2),\n            2\n        )\n\n        cart_rect = pygame.Rect(cart_x - cart_w // 2, cart_y - cart_h // 2, cart_w, cart_h)\n        pygame.draw.rect(self.screen, (30, 144, 255), cart_rect)\n        pygame.draw.rect(self.screen, (0, 0, 0), cart_rect, 2)\n\n        pivot = (cart_x, cart_y - cart_h // 2)\n        pole_x = pivot[0] + pole_len * math.sin(theta)\n        pole_y = pivot[1] - pole_len * math.cos(theta)\n\n        pygame.draw.line(self.screen, (220, 80, 20), pivot, (pole_x, pole_y), 6)\n        gfxdraw.filled_circle(self.screen, int(pole_x), int(pole_y), 10, (220, 80, 20))\n        gfxdraw.aacircle(self.screen, int(pole_x), int(pole_y), 10, (0, 0, 0))\n\n        font = pygame.font.SysFont("Arial", 18)\n        txt = font.render(f"theta={theta:.2f} rad | u={self.last_u:.2f} N", True, (0, 0, 0))\n        self.screen.blit(txt, (20, 20))\n\n        if self.render_mode == "human":\n            pygame.event.pump()\n            pygame.display.flip()\n            self.clock.tick(self.metadata["render_fps"])\n        else:\n            return np.transpose(\n                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)\n            )\n\n    def close(self):\n        if self.screen is not None:\n            import pygame\n            pygame.display.quit()\n            pygame.quit()\n            self.screen = None\n            self.clock = None\n'
Path('envs/cartpole_continuous_env.py').write_text(ORIGINAL_ENV,encoding='utf-8')
print('Entorno original preparado; SHA-256:',hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest())
from envs.cartpole_continuous_env import CartPoleContinuousEnv

## 4. Recompensa R2 idéntica, reset original y pruebas de física

In [ ]:
class CartPoleRewardR1(CartPoleContinuousEnv):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        if abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)>=CFG['omega_bono_r1_rad_s']:
            r-=3.0
        return r

class CartPoleRewardR2(CartPoleRewardR1):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(x_dot)<CFG['v_bono_r2_m_s']
               and abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)<CFG['omega_bono_r1_rad_s'])
        return r+(CFG['bono_r2'] if local else 0.0)

def make_env(steps=None,force_impulse=0.0,impulse_step=None):
    return CartPoleRewardR2(max_episode_steps=steps or CFG['episode_steps'],force_max=10.0,dt=.02,
        x_threshold=2.4,noise_std=0.,viscous_friction_cart=0.,viscous_friction_pole=0.,
        random_initial_state=True,start_near_downward=True,
        external_impulse=force_impulse,impulse_step=impulse_step)

# Verificar identidad de fuerza, estado y recompensa entre entornos de ambos algoritmos.
a,b=make_env(),make_env()
for seed in [1201,8300,9990]:
    oa,_=a.reset(seed=seed);ob,_=b.reset(seed=seed)
    assert np.array_equal(oa,ob) and oa.shape==(4,)
    assert abs(abs(float(oa[2]))-math.pi)<.11
    for u in [0.0,2.0,-3.0,10.0,-10.0]:
        aa,ra,ta,ua,_=a.step(np.array([u],dtype=np.float32))
        bb,rb,tb,ub,_=b.step(np.array([u],dtype=np.float32))
        assert np.array_equal(aa,bb) and ra==rb and (ta,ua)==(tb,ub)
a.close();b.close()
assert np.isfinite(make_env()._reward(0.,0.,0.,0.,0.))
print('PRUEBAS OK: física original, R2, fuerza, cuatro estados y reset desde abajo.')

## 5. Prueba de compatibilidad SB3: DDPG y TD3 antes de gastar pasos

In [ ]:
def create_model(algo,env,seed):
    assert algo in ALGOS
    set_random_seed(int(seed))
    noise=NormalActionNoise(mean=np.zeros(1),sigma=np.ones(1)*H02['sigma'])
    common=dict(learning_rate=H02['learning_rate'],gamma=H02['gamma'],tau=H02['tau'],
        batch_size=H02['batch_size'],buffer_size=100000,
        learning_starts=CFG['learning_starts'],train_freq=(1,'step'),gradient_steps=1,
        action_noise=noise,seed=int(seed),device='cpu',verbose=0)
    if algo=='TD3_H02_R2':
        return TD3('MlpPolicy',env,policy_kwargs={'net_arch':H02['net_arch'],'n_critics':2},
                   **TD3_SPEC,**common)
    return DDPG('MlpPolicy',env,policy_kwargs={'net_arch':H02['net_arch'],'n_critics':1},**common)

def load_model(algo,path):
    return (TD3 if algo=='TD3_H02_R2' else DDPG).load(str(path),device='cpu')

for algo in ALGOS:
    e=Monitor(make_env(100))
    m=create_model(algo,e,1301)
    o,_=e.reset(seed=8400)
    a,_=m.predict(o,deterministic=True)
    assert a.shape==(1,) and np.isfinite(a).all() and abs(float(a[0]))<=10.000001
    assert m.action_space.shape==(1,) and m.observation_space.shape==(4,)
    assert m.policy.critic.n_critics==(2 if algo.startswith('TD3') else 1)
    if algo.startswith('TD3'):
        assert m.policy_delay==2 and np.isclose(m.target_policy_noise,.20)
    m.learn(total_timesteps=4,progress_bar=False)
    assert m.num_timesteps==4
    e.close()
print('PRUEBAS OK: instanciación, críticos, dimensiones, inferencia y minibucle de ambos algoritmos.')

## 6. Evaluación común: cuatro restricciones estrictas y perturbaciones opcionales

In [ ]:
def evaluar(modelo,seed,steps=None,force_impulse=0.,impulse_step=None,guardar_traza=False):
    steps=steps or CFG['episode_steps']
    env=make_env(steps,force_impulse,impulse_step)
    obs,_=env.reset(seed=int(seed))
    largo=actual=estable=cerca=0
    cumplen=dict(x=0,v=0,theta=0,omega=0)
    sum_theta_sq=sum_control_sq=0.0
    traza=[]
    for k in range(steps):
        accion,_=modelo.predict(obs,deterministic=True)
        u=float(np.asarray(accion).reshape(-1)[0])
        obs,_,term,trunc,info=env.step(np.array([u],dtype=np.float32))
        x,v,th,w=map(float,env.unwrapped.state)
        dentro=(abs(x)<=.20 and abs(v)<=.10 and abs(th)<=.05 and abs(w)<=.10)
        actual=actual+1 if dentro else 0
        largo=max(largo,actual);estable+=int(dentro);cerca+=int(abs(th)<.15)
        for key,valor,limite in [('x',x,.20),('v',v,.10),('theta',th,.05),('omega',w,.10)]:
            cumplen[key]+=int(abs(valor)<=limite)
        sum_theta_sq+=th*th;sum_control_sq+=u*u*CFG['dt']
        if guardar_traza:
            traza.append(dict(t=(k+1)*CFG['dt'],x=x,v=v,theta=th,omega=w,u=u,estricta=int(dentro)))
        if term or trunc:break
    n=k+1; env.close()
    res=dict(estabilidad_max_s=largo*CFG['dt'],ocupacion_pct=100*estable/n,
        supervivencia_s=n*CFG['dt'],rmse_theta_rad=math.sqrt(sum_theta_sq/n),
        esfuerzo_N2s=sum_control_sq,fraccion_cerca_vertical=cerca/n,
        fin_por_riel=bool(term),
        **{f'cumple_{key}_pct':100*valor/n for key,valor in cumplen.items()})
    return (res,pd.DataFrame(traza)) if guardar_traza else res

class ZeroPolicy:
    def predict(self,obs,deterministic=True):return np.array([0.],dtype=np.float32),None
q=evaluar(ZeroPolicy(),10010,100)
assert q['estabilidad_max_s']<=q['supervivencia_s'] and q['ocupacion_pct']<=100
assert set(k for k in q if k.startswith('cumple_'))=={'cumple_x_pct','cumple_v_pct','cumple_theta_pct','cumple_omega_pct'}
print('PRUEBAS OK: evaluación determinista sin currículo ni ruido de exploración.')

## 7. Entrenamiento persistente: diez ramas, cuatro checkpoints cada una

In [ ]:
def write_json_atomic(path,data):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.partial')
    tmp.write_text(json.dumps(data,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf8')
    tmp.replace(path)

def save_model_atomic(model,destination):
    destination=Path(destination)
    tmp=destination.with_name(destination.stem+'.partial.zip')
    model.save(str(tmp))
    assert tmp.exists() and tmp.stat().st_size>1000
    with zipfile.ZipFile(tmp) as z: assert z.testzip() is None
    tmp.replace(destination)

class CheckpointCallback(BaseCallback):
    def __init__(self,folder):
        super().__init__();self.folder=Path(folder)
    def _on_step(self):
        t=int(self.model.num_timesteps)
        if t in CFG['checkpoints']:
            dest=self.folder/f'modelo_{t}.zip'
            save_model_atomic(self.model,dest)
            print(f'  Checkpoint {t:,}: {dest.name}',flush=True)
        return True

def entrenar(seed,algo):
    folder=OUT/f'seed_{seed}'/algo;folder.mkdir(parents=True,exist_ok=True)
    regfile=folder/'registro.json'
    expected=[folder/f'modelo_{x}.zip' for x in CFG['checkpoints']]
    if regfile.exists():
        reg=json.loads(regfile.read_text(encoding='utf8'))
        if (reg.get('completed') and reg.get('seed')==seed and reg.get('algo')==algo
            and reg.get('steps')==CFG['total_steps'] and reg.get('H02')==H02
            and reg.get('TD3_spec')==(TD3_SPEC if algo.startswith('TD3') else None)
            and all(p.exists() and p.stat().st_size>1000 for p in expected)):
            print('REUTILIZADO:',seed,algo,flush=True);return
    set_random_seed(int(seed))
    env=Monitor(make_env(),filename=str(folder/'monitor'))
    env.reset(seed=int(seed))
    model=create_model(algo,env,seed)
    print('ENTRENANDO:',seed,algo,flush=True)
    t0=time.perf_counter()
    try:
        model.learn(total_timesteps=CFG['total_steps'],callback=CheckpointCallback(folder),progress_bar=False)
        assert model.num_timesteps==CFG['total_steps'] and all(p.exists() for p in expected)
    finally:
        env.close()
    write_json_atomic(regfile,dict(completed=True,seed=int(seed),algo=algo,
        steps=CFG['total_steps'],H02=H02,TD3_spec=(TD3_SPEC if algo.startswith('TD3') else None),
        minutes=(time.perf_counter()-t0)/60,environment='CartPoleRewardR2 original N5.1, original downward reset'))
    print('COMPLETADO:',seed,algo,flush=True)

print('Checkpoints existentes:')
for seed in CFG['train_seeds']:
    for algo in ALGOS:
        f=OUT/f'seed_{seed}'/algo
        print(seed,algo,[k for k in CFG['checkpoints'] if (f/f'modelo_{k}.zip').exists()])
for seed in CFG['train_seeds']:
    for algo in ALGOS:
        entrenar(seed,algo)
print('ENTRENAMIENTOS COMPLETOS: 10/10')

## 8. Validación de todos los checkpoints — no usar test para selección

In [ ]:
expected=[OUT/f'seed_{seed}'/algo/f'modelo_{cp}.zip' for seed in CFG['train_seeds']
          for algo in ALGOS for cp in CFG['checkpoints']]
assert all(p.exists() and p.stat().st_size>1000 for p in expected)
rows=[]
for seed in CFG['train_seeds']:
    for algo in ALGOS:
        for cp in CFG['checkpoints']:
            model=load_model(algo,OUT/f'seed_{seed}'/algo/f'modelo_{cp}.zip')
            assert model.num_timesteps==cp
            for valseed in CFG['val_seeds']:
                vals=evaluar(model,valseed)
                rows.append(dict(train_seed=seed,algoritmo=algo,checkpoint=cp,eval_seed=valseed,**vals))
valid=pd.DataFrame(rows)
assert len(valid)==len(CFG['train_seeds'])*len(ALGOS)*len(CFG['checkpoints'])*len(CFG['val_seeds'])
valid.to_csv(OUT/'validacion_checkpoints.csv',index=False)
summary=valid.groupby(['train_seed','algoritmo','checkpoint'],as_index=False)[
    ['estabilidad_max_s','ocupacion_pct','fraccion_cerca_vertical','rmse_theta_rad','cumple_v_pct']].mean()
summary.to_csv(OUT/'resumen_validacion.csv',index=False)
selected=(summary.sort_values(['train_seed','algoritmo','estabilidad_max_s','ocupacion_pct',
        'fraccion_cerca_vertical','rmse_theta_rad','cumple_v_pct','checkpoint'],
        ascending=[True,True,False,False,False,True,False,False],kind='stable')
    .drop_duplicates(['train_seed','algoritmo']))
assert len(selected)==len(CFG['train_seeds'])*len(ALGOS)
selected.to_csv(OUT/'seleccion_SOLO_VALIDACION.csv',index=False)
(OUT/'seleccion_SHA256.txt').write_text(hashlib.sha256((OUT/'seleccion_SOLO_VALIDACION.csv').read_bytes()).hexdigest()+'\n')
print('Checkpoints seleccionados SIN consultar el conjunto de prueba:')
print(selected.round(3).to_string(index=False))

## 9. Evaluación final reservada: mismas diez semillas para DDPG y TD3

In [ ]:
assert (OUT/'seleccion_SOLO_VALIDACION.csv').exists()
selected=pd.read_csv(OUT/'seleccion_SOLO_VALIDACION.csv')
rows=[]
for row in selected.itertuples(index=False):
    model=load_model(row.algoritmo,OUT/f'seed_{int(row.train_seed)}'/row.algoritmo/f'modelo_{int(row.checkpoint)}.zip')
    for evalseed in CFG['test_seeds']:
        guardar=(evalseed==CFG['test_seeds'][0])
        result=evaluar(model,evalseed,guardar_traza=guardar)
        if guardar:
            vals,traza=result
            traza.to_csv(OUT/f'trayectoria_{row.algoritmo}_seed_{row.train_seed}_eval_{evalseed}.csv',index=False)
        else: vals=result
        rows.append(dict(train_seed=int(row.train_seed),algoritmo=row.algoritmo,
                         checkpoint=int(row.checkpoint),eval_seed=evalseed,**vals))
test=pd.DataFrame(rows)
assert len(test)==len(CFG['train_seeds'])*len(ALGOS)*len(CFG['test_seeds'])
assert set(test.eval_seed)==set(CFG['test_seeds'])
test.to_csv(OUT/'prueba_final.csv',index=False)
metrics=['estabilidad_max_s','ocupacion_pct','supervivencia_s','rmse_theta_rad','esfuerzo_N2s',
         'cumple_x_pct','cumple_v_pct','cumple_theta_pct','cumple_omega_pct']
by_policy=test.groupby(['train_seed','algoritmo'],as_index=False)[metrics].mean()
by_policy.to_csv(OUT/'resultado_por_politica.csv',index=False)
pivot=by_policy.pivot(index='train_seed',columns='algoritmo',values='estabilidad_max_s')
delta=(pivot['TD3_H02_R2']-pivot['DDPG_H02_R2']).rename('TD3_menos_DDPG_s')
delta.to_csv(OUT/'delta_pareado_por_semilla.csv',header=True)
print('RESULTADOS POR POLÍTICA:')
print(by_policy.round(3).to_string(index=False))
print('DELTA ESTABILIDAD TD3-DDPG:',delta.round(3).to_dict(),'; media:',round(float(delta.mean()),3))
print('ADVERTENCIA: cinco semillas de ENTRENAMIENTO; 50 episodios de test por algoritmo NO equivalen a 50 entrenamientos independientes.')

## 10. Contraste estadístico emparejado por semilla (solo después de evaluar)

In [ ]:
# Contraste descriptivo y exacto por SEMILLA, tras cerrar la evaluación final.
import itertools
pairs=pivot[['DDPG_H02_R2','TD3_H02_R2']].copy().sort_index()
differences=(pairs['TD3_H02_R2']-pairs['DDPG_H02_R2']).to_numpy(dtype=float)
mean_delta=float(np.mean(differences))
# Sign-flip exacto de la media emparejada (bilateral), sin tratar episodios como independientes.
null_means=np.asarray([np.mean(differences*np.asarray(signs))
    for signs in itertools.product((-1.,1.),repeat=len(differences))])
p_exact=float(np.mean(np.abs(null_means)>=abs(mean_delta)-1e-12))
rng=np.random.default_rng(20261112)
boots=np.mean(rng.choice(differences,size=(20000,len(differences)),replace=True),axis=1)
lo,hi=np.quantile(boots,[.025,.975]).tolist()
result=dict(n_pares=int(len(differences)),media_TD3_menos_DDPG_s=mean_delta,
    mediana_diferencia_s=float(np.median(differences)),
    pares_positivos=int(np.sum(differences>0)),pares_negativos=int(np.sum(differences<0)),
    pares_iguales=int(np.sum(differences==0)),p_inversion_signos_bilateral=p_exact,
    bootstrap_percentil95_inferior_s=float(lo),bootstrap_percentil95_superior_s=float(hi),
    unidad_inferencia='semilla de entrenamiento; cinco pares',
    advertencia='intervalo bootstrap descriptivo con n=5; p exacto bilateral mínimo 0.0625 para cinco diferencias no nulas')
write_json_atomic(OUT/'analisis_estadistico_por_semilla.json',result)
print('ANÁLISIS ESTADÍSTICO POR SEMILLA:',json.dumps(result,indent=2,ensure_ascii=False))
print('PRUEBAS OK: cinco pares; no se usaron episodios como muestras independientes.')


## 11. Perturbaciones: evaluación secundaria sin seleccionar nueva configuración

In [ ]:
# Pulsos de UN PASO de simulación, como el entorno N5.1 original.
# Estas condiciones son diagnósticas y NO se usan para cambiar el checkpoint elegido.
rows=[]
for row in selected.itertuples(index=False):
    model=load_model(row.algoritmo,OUT/f'seed_{int(row.train_seed)}'/row.algoritmo/f'modelo_{int(row.checkpoint)}.zip')
    for force in (0.,2.,5.,10.):
        for evalseed in CFG['test_seeds'][:3]:
            vals=evaluar(model,evalseed,force_impulse=force,impulse_step=1000)
            rows.append(dict(train_seed=int(row.train_seed),algoritmo=row.algoritmo,
                             perturbacion_N=force,duracion_s=CFG['dt'],eval_seed=evalseed,**vals))
pert=pd.DataFrame(rows)
assert len(pert)==len(CFG['train_seeds'])*len(ALGOS)*4*3
pert.to_csv(OUT/'diagnostico_perturbacion_un_paso.csv',index=False)
print(pert.groupby(['algoritmo','perturbacion_N'])['estabilidad_max_s'].agg(['mean','std']).round(2))
print('Nota: pulso de un paso (0,02 s); no comparable directamente con pulsos H1-H3 de 0,2/1 s.')

## 12. Gráficos, manifiesto y ZIP descargable

In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5))
for algo,grupo in by_policy.groupby('algoritmo'):
    ax.scatter(grupo['train_seed'].astype(str),grupo['estabilidad_max_s'],label=algo,s=65)
ax.set_xlabel('Semilla independiente de entrenamiento');ax.set_ylabel('Estabilidad máxima continua media (s)')
ax.set_ylim(0,120);ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(OUT/'comparacion_semillas.png',dpi=160);plt.show()
manifest=dict(titulo='TD3-02 confirmacion independiente TD3 vs DDPG H02-R2',
    enfoque='replica independiente de 5 semillas con configuracion congelada',CFG=CFG,H02=H02,TD3=TD3_SPEC,
    version_SB3=sb3.__version__,version_gymnasium=gym.__version__,python=platform.python_version(),
    entorno_sha256=hashlib.sha256(ORIGINAL_ENV.encode('utf8')).hexdigest(),
    seleccion_sha256=hashlib.sha256((OUT/'seleccion_SOLO_VALIDACION.csv').read_bytes()).hexdigest(),
    advertencia='TD3 no sustituye DDPG como nucleo de la tesis; 5 semillas tienen potencia estadistica limitada')
write_json_atomic(OUT/'manifiesto.json',manifest)
zip_path=Path('/content/TD3_02_VALIDACION_INDEPENDIENTE_RESULTADOS.zip')
include=[p for p in OUT.rglob('*') if p.is_file() and not p.name.endswith('.partial')]
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in include:z.write(p,p.relative_to(OUT))
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
print('ZIP:',zip_path,'archivos:',len(include),'bytes:',zip_path.stat().st_size)
files.download(str(zip_path))

## Interpretación y criterios de lectura

Esta campaña es una **réplica independiente** del piloto TD3-01, con configuración congelada. La unidad de comparación es la **semilla de entrenamiento** (cinco pares), no los episodios. La métrica primaria es el promedio por política de la **máxima permanencia estricta continua** en diez episodios finales de 120 s. Estudiar adicionalmente ocupación, número de políticas que superan 100 s, paso de checkpoint seleccionado, esfuerzo de control y costo computacional.

**Límite estadístico:** con 5 pares, los tests exactos bilaterales de inversión de signos tienen valor mínimo p=0,0625 si no existen diferencias nulas. Evitar equiparar `p>=.05` con ausencia demostrada de efecto; informar efecto observado, signo por semilla e intervalo de incertidumbre. No sumar TD3-01 y TD3-02 como si fueran 8 semillas prerregistradas para afirmar una confirmación independiente: esos agregados pueden ser exploratorios.

**No usar el test para elegir otra política ni ajustar hiperparámetros.** El diagnóstico de pulsos externos es secundario y dura un paso de 0,02 s, no equiparable a los escenarios prolongados H1-H3. La dinámica de aprendizaje TD3 no prueba por sí sola robustez híbrida.
